# Early Insights and Opportunities from AI Usage and Developer Trend in Algeria

In [1]:
from pathlib import Path
from urllib.request import urlopen
import json
import ssl

import altair as alt
import pandas as pd
import pycountry
import attaviz
import numpy as np
import statsmodels.formula.api as smf

attaviz.enable()


def find_project_root(marker="pyproject.toml"):
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ANTHROPIC_DIR = DATA_DIR / "AI" / "Anthropic"
GITHUB_DIR = DATA_DIR / "github"
OPENAI_DIR = DATA_DIR / "AI" / "OpenAI" / "public_release_csv"

COUNTRY_CODE = "DZ"
COUNTRY_NAME = "Algeria"

PEER_GROUPS = {
    "Regional": ["EG", "JO", "TN", "IQ"],
    "Structural": ["EC", "PE", "GH", "VN", "CO"],
    "Aspirational": ["MY", "CL", "PL", "RO"],
}
COUNTRY_NAMES = {
    "DZ": "Algeria",
    "EG": "Egypt",
    "JO": "Jordan",
    "TN": "Tunisia",
    "IQ": "Iraq",
    "EC": "Ecuador",
    "PE": "Peru",
    "GH": "Ghana",
    "VN": "Vietnam",
    "CO": "Colombia",
    "MY": "Malaysia",
    "CL": "Chile",
    "PL": "Poland",
    "RO": "Romania",
    "NG": "Nigeria",
    "KE": "Kenya",
}
GROUP_OF = {c: g for g, members in PEER_GROUPS.items() for c in members}
GROUP_OF[COUNTRY_CODE] = "Algeria"
GROUP_ORDER = ["Algeria", "Regional", "Structural", "Aspirational"]
GROUP_COLORS = {
    "Algeria": "#0071BC",
    "Regional": "#8A969F",
    "Structural": "#8A969F",
    "Aspirational": "#8A969F",
}
PEER_CODES = [COUNTRY_CODE] + [c for g in PEER_GROUPS.values() for c in g]
ALGERIA_COLOR = "#0071BC"
PEER_COLORS = ["#8A969F", "#9FA8AF", "#B0B8BE", "#C1C8CD", "#D2D7DB"]

ISO2_TO_ISO3 = {
    "DZ": "DZA",
    "EG": "EGY",
    "JO": "JOR",
    "TN": "TUN",
    "IQ": "IRQ",
    "EC": "ECU",
    "PE": "PER",
    "GH": "GHA",
    "VN": "VNM",
    "CO": "COL",
    "MY": "MYS",
    "CL": "CHL",
    "PL": "POL",
    "RO": "ROU",
}
ISO3_TO_ISO2 = {v: k for k, v in ISO2_TO_ISO3.items()}


def country_name_from_iso2(code):
    if code == "EU":
        return "European Union"
    if code in COUNTRY_NAMES:
        return COUNTRY_NAMES[code]
    country = pycountry.countries.get(alpha_2=code)
    return country.name if country else code


REGIONAL_CODES = [COUNTRY_CODE] + PEER_GROUPS["Regional"]
GLOBAL_LABEL = "Global"
PLATFORM_COLORS = {"Claude": "#0071BC", "ChatGPT": "#FF9800"}

CHATGPT_QUARTER = "2026-04-01"
CHATGPT_MONTHS = ("2026-04-01", "2026-06-01")

In [2]:
import textwrap
from decimal import ROUND_HALF_UP, Decimal

FONT_SIZE = 13  # same as the theme's axis and legend labels
CHAR_W = FONT_SIZE * 0.62
LINE_H = FONT_SIZE * 1.35
PAD = 6
SAFETY_PX = 4
INSIDE_LABEL_THRESHOLD = 0.20


def fmt_pct(value, decimals=1):
    """Format a percentage, rounding half up so labels match the text.

    Python's f-string rounds 12.45 down to 12.4 (binary floats), but readers
    round the published two-decimal value 12.45 up to 12.5.
    """
    step = Decimal(1).scaleb(-decimals)
    rounded = Decimal(str(round(float(value), 6))).quantize(
        step, rounding=ROUND_HALF_UP
    )
    return f"{rounded}%"


def make_bar(
    data,
    label_col,
    value_col,
    title,
    subtitle=None,
    width=600,
    drop_not_classified=True,
    top_n=None,
    x_title="% of conversations",
    fmt=fmt_pct,
    group_col=None,
    colors=None,
    sort=None,
    highlight=None,
    x_max=None,
    label_top=None,
):
    """Horizontal bars with the value inside each bar (outside if short).

    Labels sit on the left, or above each bar when the longest label has more
    than 20 characters (label_top=None); pass True/False to force. `width` is
    the full chart width, so a left label column shrinks the plot.
    Single series: sorted by value. Grouped: pass group_col and colors
    ({group: color}); the bars of one label sit together in `sort` order, and
    a missing value prints "Not published". highlight colours one label blue
    and the rest grey.
    """
    d = data.copy()
    if drop_not_classified:
        d = d.loc[~d[label_col].str.contains("not_classified", case=False, na=False)]
    if group_col is None:
        d = d.loc[d[value_col] > 0].sort_values(value_col, ascending=False)
        if top_n is not None:
            d = d.head(top_n)
    groups = list(colors) if group_col else [None]
    order = sort or list(dict.fromkeys(d[label_col]))
    longest = max(len(str(v)) for v in order)
    if label_top is None:
        label_top = longest > 20
    if not label_top:
        width -= int(longest * CHAR_W) + 12  # room for the axis labels
    bar_size = 16
    gap = 26 if label_top else 12  # label above the first bar, or row spacing
    row_h = gap + (bar_size + 1) * len(groups)  # bars of one label 1px apart
    x_max = x_max or d[value_col].max() * 1.02
    inside_cutoff = x_max * INSIDE_LABEL_THRESHOLD
    plot_df = d.assign(
        _zero=0.0,
        _x=d[value_col].fillna(0),
        _value_label=d[value_col].map(
            lambda v: "Not published" if pd.isna(v) else fmt(v)
        ),
    )
    y_enc = alt.Y(
        f"{label_col}:N",
        sort=order,
        title=None,
        axis=None
        if label_top
        else alt.Axis(grid=False, labelLimit=0, labelColor="#222", labelPadding=8),
        scale=alt.Scale(paddingInner=gap / row_h, paddingOuter=0 if label_top else 0.3),
    )
    offset = {}
    if group_col:
        offset["yOffset"] = alt.YOffset(
            f"{group_col}:N", sort=groups, scale=alt.Scale(paddingInner=0)
        )
        color = alt.Color(
            f"{group_col}:N",
            title=None,
            scale=alt.Scale(domain=groups, range=list(colors.values())),
            legend=alt.Legend(orient="bottom"),
        )
    elif highlight:
        color = alt.condition(
            alt.datum[label_col] == highlight,
            alt.value(ALGERIA_COLOR),
            alt.value("#8A969F"),
        )
    else:
        color = alt.value(ALGERIA_COLOR)
    base = alt.Chart(plot_df)
    bars = base.mark_bar(size=bar_size).encode(
        x=alt.X(
            f"{value_col}:Q",
            title=x_title,
            scale=alt.Scale(domain=[0, x_max], nice=False),
        ),
        y=y_enc,
        color=color,
        tooltip=[alt.Tooltip(f"{label_col}:N", title="Group")]
        + ([alt.Tooltip(f"{group_col}:N", title="Series")] if group_col else [])
        + [alt.Tooltip("_value_label:N", title="Value")],
        **offset,
    )
    pct_inside = (
        base.transform_filter(f"datum['{value_col}'] >= {inside_cutoff}")
        .mark_text(
            align="right",
            baseline="middle",
            dx=-PAD,
            color="white",
            fontSize=FONT_SIZE - 1,
            fontWeight="bold",
        )
        .encode(x="_x:Q", y=y_enc, text="_value_label:N", **offset)
    )
    pct_outside = (
        base.transform_filter(f"!(datum['{value_col}'] >= {inside_cutoff})")
        .mark_text(align="left", baseline="middle", dx=PAD, fontSize=FONT_SIZE - 1)
        .encode(
            x="_x:Q",
            y=y_enc,
            text="_value_label:N",
            color=alt.condition(
                f"isValid(datum['{value_col}'])",
                alt.value("#444"),
                alt.value("#8A969F"),
            ),
            **offset,
        )
    )
    layers = [bars, pct_inside, pct_outside]
    if label_top:
        label_rows = (
            base.transform_filter(
                alt.FieldEqualPredicate(field=group_col, equal=groups[0])
            )
            if group_col
            else base
        )
        layers.append(
            label_rows.mark_text(
                align="left",
                baseline="bottom",
                color="#222",
                dy=-(bar_size // 2) - 2,
            ).encode(x="_zero:Q", y=y_enc, text=f"{label_col}:N", **offset)
        )
    if not isinstance(title, alt.Title):
        title = alt.Title(text=title, subtitle=subtitle or "")
    return alt.layer(*layers).properties(
        width=width, height=row_h * len(order), title=title
    )

In [3]:
# Claude.ai usage by calendar month (June 2026 release). Use the latest month.
SNAPSHOT_MONTH = "2026-05-01"
COUNTRY_ISO3 = ISO2_TO_ISO3[COUNTRY_CODE]

df = pd.read_csv(
    ANTHROPIC_DIR / "release_2026_06_26/aei_claude_ai_2026-06-26.csv",
    dtype={"geo_id": str},
).query("date_start == @SNAPSHOT_MONTH")
dza = df.query("geo_id == @COUNTRY_ISO3")

Algeria has seen the highest jump in internet connectivity in the MENAAP Region since 2019[^1], with greater internet speeds, and improved connectivity. This improvement in infrastructure came at an opportune moment for Algerians to access and harness the power of Artificial Intelligence (AI) through usage of Large Language Models (LLMs) as part of their work.

Algeria ranks 105<sup>th</sup> out of 177 countries for per-capita ChatGPT usage in the second quarter of 2026[^2]. For Claude, it ranks 82<sup>nd</sup> out of 121 countries in May 2026 on Anthropic's AI Usage Index[^3]. The index is 0.56, which means that Algeria's Claude usage is about half of what its share of the world's working-age population would suggest. Only 21.8% of Algerian Claude conversations are for work ({numref}`fig-box-use-case`). This is about half of the global share (43.4%), and the lowest among Algeria's regional peers. ChatGPT data show the same pattern: 18.0% of Algerian ChatGPT messages are for work, the second-lowest share among the regional peers after Iraq. Coursework is the opposite: it makes up 47.1% of Algerian conversations, 2.9 times the global share (16.4%). Only Tunisia is higher among the regional peers. This suggests that AI could change skill development and education in Algeria. Claude use is also concentrated in a few cities ({numref}`fig-box-wilaya`). Alger accounts for 38.5% of Algerian Claude usage, followed by Oran with 16.2% and Sétif with 12.5%. This is in line with the alternate finding that urban populations are more exposed to AI {cite:p}`oecd2024jobcreation`.

In [4]:
_iso3_names = {ISO2_TO_ISO3[code]: COUNTRY_NAMES[code] for code in REGIONAL_CODES} | {
    "GLOBAL": GLOBAL_LABEL
}

# Claude: shares renormalized over personal/work/coursework (as in the text).
_claude_use = (
    df.loc[
        lambda d: (
            (d["category_name"] == "overall")
            & d["geo_id"].isin(_iso3_names)
            & d["metric_id"].isin(
                [f"use_case_{s}_pct" for s in ["personal", "work", "coursework"]]
            )
        )
    ]
    .assign(
        share=lambda d: d.groupby("geo_id")["value"].transform(
            lambda s: s / s.sum() * 100
        )
    )
    .pivot_table(index="geo_id", columns="metric_id", values="share")
    .rename(index=_iso3_names)
)

# ChatGPT: work share only. OpenAI has no country-level schoolwork series.
_gpt_work = (
    pd.read_csv(OPENAI_DIR / "share_of_messages_by_work_related_country_month.csv")
    .query(
        "work_related == 1 and month.between(@CHATGPT_MONTHS[0], @CHATGPT_MONTHS[1])"
    )
    .groupby("country")["share_of_messages"]
    .agg(["mean", "size"])
)
_gpt_global = pd.read_csv(
    OPENAI_DIR / "share_of_messages_by_work_related_month.csv"
).query("work_related == 1 and month.between(@CHATGPT_MONTHS[0], @CHATGPT_MONTHS[1])")[
    "share_of_messages"
]
gpt_work = (
    (_gpt_work["mean"] * 100)
    .rename(index=COUNTRY_NAMES)
    .reindex([COUNTRY_NAMES[c] for c in REGIONAL_CODES])
)
gpt_work[GLOBAL_LABEL] = _gpt_global.mean() * 100

COUNTRY_ORDER = [COUNTRY_NAMES[c] for c in REGIONAL_CODES] + [GLOBAL_LABEL]
SIGNALS = ["Work", "Learning (coursework)"]
learning_work = (
    pd.concat(
        [
            _claude_use["use_case_work_pct"]
            .rename("share")
            .to_frame()
            .assign(signal="Work", platform="Claude"),
            gpt_work.rename("share")
            .to_frame()
            .assign(signal="Work", platform="ChatGPT"),
            _claude_use["use_case_coursework_pct"]
            .rename("share")
            .to_frame()
            .assign(signal=SIGNALS[1], platform="Claude"),
            pd.DataFrame({"share": float("nan")}, index=COUNTRY_ORDER).assign(
                signal=SIGNALS[1], platform="ChatGPT"
            ),
        ]
    )
    .rename_axis("country_name")
    .reset_index()
)

learning_work_chart = alt.hconcat(
    *[
        make_bar(
            learning_work.query("signal == @signal"),
            label_col="country_name",
            value_col="share",
            title=alt.Title(text=signal, fontSize=FONT_SIZE, anchor="start"),
            width=335,
            drop_not_classified=False,
            x_title="% of conversations or messages",
            group_col="platform",
            colors=PLATFORM_COLORS,
            sort=COUNTRY_ORDER,
            x_max=60,
        )
        for signal in SIGNALS
    ],
    spacing=40,
).properties(
    title=alt.Title(
        text="Work and Learning Use of Claude and ChatGPT, Regional Peers vs Global",
        subtitle=[
            "Claude: share of classified conversations, May 2026",
            "ChatGPT: share of messages, April-June 2026",
        ],
    )
)
attaviz.add_caption(
    learning_work_chart,
    "Source: Anthropic Economic Index data and OpenAI Signals data\n"
    "Note: OpenAI publishes the schoolwork share only for the world, not for each country. "
    "So the chart has no ChatGPT learning value.\n"
    "ChatGPT classifies each message as work or not work. Claude uses three groups: personal, work and coursework.",
)

alt.VConcatChart(...)

In [5]:
wilaya = (
    df.loc[
        (df["geo_level"] == "subregion")
        & (df["geo_id"].str.startswith("DZ-", na=False))
        & (df["category_name"] == "overall")
        & (df["metric_id"] == "usage_pct")
        & (df["geo_id"] != "DZ-not_classified")
    ]
    .assign(
        wilaya=lambda d: d["geo_id"].map(
            lambda c: getattr(pycountry.subdivisions.get(code=c), "name", c)
        ),
        value_label=lambda d: d["value"].map(fmt_pct),
    )[["geo_id", "wilaya", "value", "value_label"]]
    .sort_values("value", ascending=False)
    .reset_index(drop=True)
)

wilaya_bar = make_bar(
    wilaya,
    label_col="wilaya",
    value_col="value",
    title=f"Subnational Claude Usage in {COUNTRY_NAME}",
    subtitle="Share of Algeria's Claude.ai conversations by wilaya, May 2026 (wilayas with published data)",
)
attaviz.add_caption(wilaya_bar, "Source: Anthropic Economic Index data")

alt.VConcatChart(...)

Education is the most common topic of Algerian Claude conversations: 32.6% of conversations are about education and learning ({numref}`fig-box-request`). Content creation and copywriting follows with 18.6%. Software development is third, with 13.0%. When conversations are mapped to occupations, computer and mathematical occupations account for the largest share (24.9%), followed by educational instruction and library occupations (20.0%) ({numref}`fig-box-occupation`). The occupation mapping describes the type of task, not the job of the user. Because most Algerian conversations are not about work, these results suggest that Algerians use Claude mainly to learn and to write. Many are also building software skills, either for study or for work.

In [6]:
def request_df(level):
    return (
        dza.loc[
            lambda df: (
                (df["category_name"] == "request")
                & (df["metric_id"] == "pct")
                & (df["hierarchy_level"] == level)
            )
        ]
        .filter(["node_name", "value"])
        .rename(columns={"node_name": "cluster_name", "value": "pct"})
        .sort_values("pct", ascending=False)
        .reset_index(drop=True)
    )


req_l2 = request_df(2)

In [7]:
l2_bar_chart = make_bar(
    req_l2,
    label_col="cluster_name",
    value_col="pct",
    title=f"Claude Usage by Request Category in {COUNTRY_NAME}",
    subtitle="Request category derived from messages, May 2026 release",
    top_n=10,
)
attaviz.add_caption(l2_bar_chart, "Source: Anthropic Economic Index data")

alt.VConcatChart(...)

In [8]:
occupation_df = (
    dza.loc[
        lambda df: (
            (df["category_name"] == "soc_occupation")
            & (df["metric_id"] == "pct")
            & (df["hierarchy_level"] == 1)
        )
    ]
    .filter(["node_name", "value"])
    .rename(columns={"node_name": "soc_name", "value": "pct"})
    .sort_values("pct", ascending=False)
    .reset_index(drop=True)
)

In [9]:
occupation_bar_chart = make_bar(
    occupation_df,
    label_col="soc_name",
    value_col="pct",
    title=f"Claude Usage by Occupation Group in {COUNTRY_NAME}",
    subtitle="Conversations mapped to O*NET-SOC occupation groups, May 2026 release",
    top_n=10,
)
attaviz.add_caption(occupation_bar_chart, "Source: Anthropic Economic Index data")

alt.VConcatChart(...)

Claude and ChatGPT are used for different kinds of work in Algeria ({numref}`fig-box-work-activities`). When only work use is compared, working with computers accounts for 22.3% of Algerian Claude work use, but only 5.8% of ChatGPT work use. ChatGPT work use leans more toward making decisions and solving problems (17.2%, compared with 6.4% on Claude), getting information (13.1%, compared with 6.2%), and interpreting information for others, which includes translation (10.2%, compared with 3.3%). Creative thinking, which includes writing, is among the largest activities on both platforms. So Algerian work use of Claude is more concentrated in software tasks, and ChatGPT work use is spread across writing, information and translation tasks.

In [10]:
# O*NET Generalized Work Activities (GWA), work use only, on both platforms.
#
# Why GWA: Claude publishes IWA shares for a country only over ALL conversations
# (work, personal and coursework); its work split exists only at GWA level.
# ChatGPT publishes IWA shares for work messages only. GWA is the finest level
# where both platforms have a work-only figure.
#
# Claude, per GWA (README of the 2026-06-26 release):
#   pct               = GWA's share of ALL the country's conversations
#   use_case_work_pct = share of THIS GWA's conversations that are work
# The README does not name the denominator of use_case_work_pct; the checks
# below show it is the GWA's own conversations. So each work share has a
# different base and the shares do not add up to 100 across GWAs. The product
#   pct * use_case_work_pct / 100
# puts every GWA on one base: its work conversations as a share of ALL the
# country's conversations. These values can be added and rescaled.
#
# ChatGPT: IWA shares are already work-only; roll IWAs up to GWAs by ID prefix
# (IWA 4.A.3.b.6.I01 belongs to GWA 4.A.3.b.6).
_gwa = df.query(
    "category_name == 'onet' and hierarchy_level == 3 and geo_id == @COUNTRY_ISO3"
).pivot_table(index="node_external_id", columns="metric_id", values="value")
claude_gwa_work = _gwa["pct"] * _gwa["use_case_work_pct"] / 100

_gpt_iwa = pd.read_csv(
    OPENAI_DIR / "share_of_work_related_messages_by_onet_iwa_country_quarter.csv"
).query("quarter == @CHATGPT_QUARTER and country == @COUNTRY_CODE")
gpt_other_iwa = (
    _gpt_iwa.query("iwa_cleaned == 'Other IWA'")["share_of_messages"].sum() * 100
)
_gpt_iwa = _gpt_iwa.query("iwa_cleaned != 'Other IWA'")
gpt_gwa_work = (
    _gpt_iwa.assign(gwa=lambda df: df["iwa_cleaned"].str.split(".I").str[0])
    .groupby("gwa")["share_of_messages"]
    .sum()
)

GWA_NAME = (
    df.query("category_name == 'onet' and hierarchy_level == 3")
    .drop_duplicates("node_external_id")
    .set_index("node_external_id")["node_name"]
)
_both = pd.concat(
    [claude_gwa_work.rename("Claude"), gpt_gwa_work.rename("ChatGPT")], axis=1
)
gpt_only_share = (
    _both.loc[_both["Claude"].isna(), "ChatGPT"].sum() / _both["ChatGPT"].sum() * 100
)
gwa_work = _both.dropna()
gwa_work = gwa_work / gwa_work.sum() * 100  # same GWA set on both sides
gwa_work["activity"] = gwa_work.index.map(GWA_NAME)
_r = gwa_work.set_index("activity").round(1)

# Show activities with at least 1% on either platform.
_shown = gwa_work.loc[gwa_work[["Claude", "ChatGPT"]].max(axis=1) >= 1].sort_values(
    "Claude", ascending=False
)
GWA_ORDER = _shown["activity"].tolist()
gwa_long = _shown.melt(id_vars="activity", var_name="platform", value_name="share")
work_activities_chart = make_bar(
    gwa_long,
    label_col="activity",
    value_col="share",
    title=f"Work Activities in {COUNTRY_NAME}, Claude vs ChatGPT",
    subtitle=[
        "O*NET Generalized Work Activities, share of work use",
        "Claude: May 2026. ChatGPT: April-June 2026",
    ],
    drop_not_classified=False,
    x_title="% of work use",
    group_col="platform",
    colors=PLATFORM_COLORS,
    sort=GWA_ORDER,
)
attaviz.add_caption(
    work_activities_chart,
    "Source: Anthropic Economic Index data and OpenAI Signals data\n"
    "Note: Both platforms are on a work-only base. Claude labels all conversations, so the chart keeps only the work share of each activity.\n"
    "Claude publishes IWA shares only for all conversations, and the work split only at GWA level. ChatGPT publishes IWA shares for work messages only.\nThe chart therefore compares the two at GWA level. "
    f'OpenAI groups rare IWAs as "Other IWA" ({gpt_other_iwa:.1f}%), which the chart excludes.\n'
    f"Shares are rescaled over the {len(gwa_work)} activities on both platforms. The chart shows activities with at least 1% on either platform.",
)

alt.VConcatChart(...)

## Underutilized, but rising software development skills in Algeria

Algerians use AI widely to learn, but this use has not yet turned into work or into software. Across 119 countries, Claude usage rises closely with the number of GitHub developers per working-age person ({numref}`fig-box-ai-developers`, left panel). Algeria is in line with this trend, as the country's Claude usage matches what its developer base predicts[^4]. In the second quarter of 2026, Algerian developers made 1.08 pushes each on average, the second lowest among peers after Iraq. The constraint is therefore not interest in AI, but rather the number and the activity of the people who build software.

In [11]:
GH_PEER_GROUPS = {
    "Developers": ["EG", "NG", "KE"],
    "Structural": PEER_GROUPS["Structural"],
    "Regional": ["EG", "JO", "TN", "IQ"],
    "Aspirational": ["MY", "CL", "PL", "RO"],
}
GH_PEER_CODES = [COUNTRY_CODE] + sorted(
    {code for codes in GH_PEER_GROUPS.values() for code in codes}
)


def _iso3_to_iso2(code):
    country = pycountry.countries.get(alpha_3=code)
    return country.alpha_2 if country else None


def load_working_age_population():
    """Working-age population (SP.POP.1564.TO) for every economy, fetched once."""
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    url = (
        "https://api.worldbank.org/v2/country/all"
        "/indicator/SP.POP.1564.TO?format=json&per_page=20000"
    )
    with urlopen(url, context=ctx) as response:
        payload = json.load(response)
    return (
        pd.DataFrame(payload[1])
        .loc[:, ["countryiso3code", "date", "value"]]
        .rename(
            columns={
                "countryiso3code": "iso3_code",
                "date": "year",
                "value": "working_age_population",
            }
        )
        .dropna(subset=["working_age_population"])
        .assign(
            iso2_code=lambda d: d["iso3_code"].map(_iso3_to_iso2),
            year=lambda d: d["year"].astype(int),
            working_age_population=lambda d: d["working_age_population"].astype(float),
        )
        .dropna(subset=["iso2_code"])
        .sort_values(["iso2_code", "year"])
    )


def attach_latest_population(github_df, population_df):
    pieces = []
    for iso2_code, country_df in github_df.sort_values("year").groupby(
        "iso2_code", sort=False
    ):
        pop_country = population_df.loc[
            population_df["iso2_code"] == iso2_code
        ].sort_values("year")
        if pop_country.empty:
            continue
        pieces.append(
            pd.merge_asof(
                country_df.sort_values("year"),
                pop_country[["year", "working_age_population"]],
                on="year",
                direction="backward",
            )
        )
    return pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame()


def load_github_metric(metric, population_df, peer_codes):
    raw = (
        pd.read_csv(GITHUB_DIR / f"{metric}.csv")
        .dropna(subset=["iso2_code"])
        .loc[lambda d: d["iso2_code"] != "EU"]
        .assign(
            iso2_code=lambda d: d["iso2_code"].astype(str),
            quarter_start=lambda d: pd.PeriodIndex(
                d["year"].astype(str) + "Q" + d["quarter"].astype(str), freq="Q"
            ).to_timestamp(),
        )
        .sort_values(["iso2_code", "year", "quarter"])
    )
    per_100k = f"{metric}_per_100k"
    return (
        attach_latest_population(raw, population_df)
        .assign(
            **{per_100k: lambda d: d[metric] / d["working_age_population"] * 100_000}
        )
        .dropna(subset=[per_100k])
        .loc[lambda d: d["iso2_code"].isin(peer_codes)]
        .assign(
            country_name=lambda d: (
                d["iso2_code"]
                .map(COUNTRY_NAMES)
                .fillna(d["iso2_code"].map(country_name_from_iso2))
            )
        )
        .sort_values(["iso2_code", "year", "quarter"])
    )


working_age_population = load_working_age_population()
developers = load_github_metric("developers", working_age_population, GH_PEER_CODES)
git_pushes = load_github_metric("git_pushes", working_age_population, GH_PEER_CODES)
clusters = pd.read_csv(GITHUB_DIR / "language_to_cluster_mapping.csv")

# Latest GitHub quarter.
GH_YEAR, GH_QUARTER = developers.sort_values(["year", "quarter"])[
    ["year", "quarter"]
].iloc[-1]
GH_LABEL = f"{GH_YEAR} Q{GH_QUARTER}"

In [12]:
_aui = (
    df.loc[
        lambda d: (
            (d["geo_level"] == "country")
            & (d["category_name"] == "overall")
            & (d["metric_id"] == "usage_per_capita_index")
        )
    ]
    .assign(iso2_code=lambda d: d["geo_id"].map(_iso3_to_iso2))
    .set_index("iso2_code")["value"]
    .rename("aui")
)
_dev_all = load_github_metric("developers", working_age_population, set(_aui.index))
_dev_latest = _dev_all.query("year == @GH_YEAR and quarter == @GH_QUARTER").set_index(
    "iso2_code"
)["developers_per_100k"]
_push_latest = (
    load_github_metric(
        "git_pushes",
        working_age_population,
        GH_PEER_CODES,
    )
    .query("year == @GH_YEAR and quarter == @GH_QUARTER")
    .set_index("iso2_code")["git_pushes"]
)
ai_dev = (
    pd.concat([_aui, _dev_latest.rename("dev_per_100k")], axis=1, join="inner")
    .reset_index(names="iso2_code")
    .assign(
        country_name=lambda d: d["iso2_code"].map(country_name_from_iso2),
        group=lambda d: (
            d["iso2_code"]
            .map(GROUP_OF)
            .fillna(
                d["iso2_code"].map(lambda c: "Peer" if c in ("NG", "KE") else "Other")
            )
        ),
    )
    .replace(
        {"group": {"Regional": "Peer", "Structural": "Peer", "Aspirational": "Peer"}}
    )
)

_ppd = (
    _push_latest
    / _dev_all.query("year == @GH_YEAR and quarter == @GH_QUARTER").set_index(
        "iso2_code"
    )["developers"]
).dropna()
_ppd = _ppd.loc[_ppd.index.isin(PEER_CODES + ["NG", "KE"])].sort_values()

# ChatGPT has only a per-capita RANK. Convert it to a percentile (100 = top).
_gpt_rank = (
    pd.read_csv(OPENAI_DIR / "share_of_messages_by_country_quarter_rank.csv")
    .query("quarter == @CHATGPT_QUARTER")
    .drop_duplicates("country")
    .set_index("country")["rank"]
)
N_CHATGPT = len(_gpt_rank)

# Developers for every ChatGPT-ranked country, not only those Claude publishes.
_gpt_dev_latest = (
    load_github_metric("developers", working_age_population, set(_gpt_rank.index))
    .query("year == @GH_YEAR and quarter == @GH_QUARTER")
    .set_index("iso2_code")["developers_per_100k"]
)
gpt_dev = (
    pd.concat(
        [
            (100 * (1 - (_gpt_rank - 1) / (N_CHATGPT - 1))).rename("gpt_pctile"),
            _gpt_rank.rename("gpt_rank"),
            _gpt_dev_latest.rename("dev_per_100k"),
        ],
        axis=1,
        join="inner",
    )
    .reset_index(names="iso2_code")
    .assign(
        country_name=lambda d: d["iso2_code"].map(country_name_from_iso2),
        group=lambda d: d["iso2_code"].map(
            lambda c: (
                "Algeria"
                if c == COUNTRY_CODE
                else "Peer"
                if c in PEER_CODES + ["NG", "KE"]
                else "Other"
            )
        ),
    )
)

_gfit = smf.ols("gpt_pctile ~ np.log(dev_per_100k)", data=gpt_dev).fit()
gpt_dz_resid = _gfit.resid[gpt_dev["iso2_code"] == "DZ"].item()

In [13]:
# Shared by Charts 2a and 2b so the pair reads on the same x-axis.
DEV_X = alt.X(
    "dev_per_100k:Q",
    scale=alt.Scale(type="log", domain=[20, 60_000], nice=False),
    axis=alt.Axis(
        values=[20, 100, 500, 2_000, 10_000, 50_000],
        # Explicit labels: the theme's tickCount otherwise hides some log ticks.
        labelExpr="format(datum.value, ',')",
        # The theme's tickCount (5) thins log-axis ticks; keep all six.
        tickCount=6,
        labelOverlap=False,
    ),
    title="GitHub developers per 100k (log scale)",
)
SCATTER_LABELS = {
    "DZ": ("right", -9),
    "EG": ("left", 8),
    "TN": ("right", -9),
    "PL": ("left", 8),
}
_y = alt.Y(
    "aui:Q",
    scale=alt.Scale(type="log", domain=[0.05, 10], nice=False),
    axis=alt.Axis(
        values=[0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10], tickCount=8, labelOverlap=False
    ),
    title="Claude AI Usage Index (log scale)",
)
_fit = smf.ols("np.log(aui) ~ np.log(dev_per_100k)", data=ai_dev).fit()
_line = pd.DataFrame(
    {
        "dev_per_100k": np.exp(
            np.linspace(*np.log(ai_dev["dev_per_100k"]).agg(["min", "max"]), 50)
        )
    }
).assign(aui=lambda d: np.exp(_fit.predict(d)))
_base = alt.Chart(ai_dev).encode(
    x=DEV_X,
    y=_y,
    tooltip=[
        alt.Tooltip("country_name:N", title="Country"),
        alt.Tooltip("aui:Q", title="Usage Index", format=".2f"),
        alt.Tooltip("dev_per_100k:Q", title="Developers per 100k", format=",.0f"),
    ],
)
claude_panel = alt.layer(
    _base.transform_filter("datum.group == 'Other'").mark_circle(
        size=40, color="#D2D7DB", opacity=1
    ),
    _base.transform_filter("datum.group == 'Peer'").mark_circle(
        size=70, color="#8A969F", opacity=1
    ),
    alt.Chart(_line)
    .mark_line(color="#8A969F", strokeDash=[6, 4], strokeWidth=1.5, clip=True)
    .encode(x=DEV_X, y=_y),
    _base.transform_filter("datum.group == 'Algeria'").mark_circle(
        size=120, color=ALGERIA_COLOR, opacity=1
    ),
    *[
        _base.transform_filter(f"datum.iso2_code == '{code}'")
        .mark_text(
            align=align,
            dx=dx,
            fontSize=12,
            fontWeight="bold" if code == "DZ" else "normal",
            color=ALGERIA_COLOR if code == "DZ" else "#666666",
        )
        .encode(text="country_name:N")
        for code, (align, dx) in SCATTER_LABELS.items()
    ],
).properties(
    width=265,
    height=300,
    title=alt.Title(text="Claude", fontSize=FONT_SIZE, anchor="start"),
)

# ChatGPT panel: same x-axis; y is a percentile because OpenAI publishes only a rank.
_y = alt.Y(
    "gpt_pctile:Q",
    scale=alt.Scale(domain=[0, 100]),
    title="ChatGPT usage percentile (100 = highest)",
)
_fit = smf.ols("gpt_pctile ~ np.log(dev_per_100k)", data=gpt_dev).fit()
_line = pd.DataFrame(
    {
        "dev_per_100k": np.exp(
            np.linspace(*np.log(gpt_dev["dev_per_100k"]).agg(["min", "max"]), 50)
        )
    }
).assign(gpt_pctile=lambda d: _fit.predict(d))
_base = alt.Chart(gpt_dev).encode(
    x=DEV_X,
    y=_y,
    tooltip=[
        alt.Tooltip("country_name:N", title="Country"),
        alt.Tooltip("gpt_rank:Q", title="ChatGPT rank"),
        alt.Tooltip("dev_per_100k:Q", title="Developers per 100k", format=",.0f"),
    ],
)
chatgpt_panel = alt.layer(
    _base.transform_filter("datum.group == 'Other'").mark_circle(
        size=40, color="#D2D7DB", opacity=1
    ),
    _base.transform_filter("datum.group == 'Peer'").mark_circle(
        size=70, color="#8A969F", opacity=1
    ),
    alt.Chart(_line)
    .mark_line(color="#8A969F", strokeDash=[6, 4], strokeWidth=1.5, clip=True)
    .encode(x=DEV_X, y=_y),
    _base.transform_filter("datum.group == 'Algeria'").mark_circle(
        size=120, color=ALGERIA_COLOR, opacity=1
    ),
    *[
        _base.transform_filter(f"datum.iso2_code == '{code}'")
        .mark_text(
            align=align,
            dx=dx,
            fontSize=12,
            fontWeight="bold" if code == "DZ" else "normal",
            color=ALGERIA_COLOR if code == "DZ" else "#666666",
        )
        .encode(text="country_name:N")
        for code, (align, dx) in SCATTER_LABELS.items()
    ],
).properties(
    width=265,
    height=300,
    title=alt.Title(text="ChatGPT", fontSize=FONT_SIZE, anchor="start"),
)

ai_dev_chart = alt.hconcat(claude_panel, chatgpt_panel, spacing=40).properties(
    title=alt.Title(
        text="AI Usage and GitHub Developer Density, Claude and ChatGPT",
        subtitle=[
            "Dashed line represents fit on developer density, and dark grey represents peer countries.",
        ],
    )
)
attaviz.add_caption(
    ai_dev_chart,
    f"Source: Anthropic Economic Index (May 2026), OpenAI Signals data (2026 Q2), and GitHub Innovation Graph ({GH_LABEL})\n"
    "Note: OpenAI publishes only a rank of messages per capita. The ChatGPT percentile rescales the rank of 177 countries to 0-100.",
)

alt.VConcatChart(...)

ChatGPT use follows the same pattern ({numref}`fig-box-ai-developers`, right panel). Across 170 countries, countries with more GitHub developers per working-age person rank higher for ChatGPT use. Algeria's ChatGPT rank is close to what its developer base predicts.

This supply of builders is growing quickly. On the open-source software platform GitHub, the number of developers increased from about 330 to 2,200 per 100,000 working-age people--with a clear inflection point seen with the launch of free access AI models through GitHub copilot ({numref}`fig-box-developers`). A similar increase is seen across Sub-Saharan Africa, MENAAP and South Asia Region (reference). Algeria now has more developers per working-age person than Nigeria and about as many as Kenya, but fewer than most of its structural, regional and aspirational peers.

As of 2025, LinkedIn data suggests that the disruptive tech skill penetration in Algeria is lower than most of its regional peers ({numref}`fig-box-linkedin`). This includes skills such as Artificial Intelligence, robotics, and other skills that are meant to disrupt the labor markets in the coming years. This gap in Algeria’s skill could be bridged, in some part by the rising developer trend, and by using competence in software to its advantage.

In [14]:
ALL_TREND_LABEL_CODES = ["DZ", "PL", "TN", "NG"]
ALL_TREND_GROUP_ORDER = [
    "Developers peers",
    "Kenya",
    "Structural peers",
    "Regional peers",
    "Aspirational peers",
    "Algeria",
]
ALL_TREND_GROUP_COLORS = [
    "#4EC2C0",
    "#0C7C68",
    "#F3578E",
    "#FF9800",
    "#664AB6",
    ALGERIA_COLOR,
]


def make_all_peer_trend(data, raw_col, metric_label, width=560, height=380):
    value_col = f"{raw_col}_per_100k"
    y_axis_title = f"{metric_label}/100k"
    title = f"GitHub {metric_label} per 100k Working-Age Population"
    subtitle = f"Algeria and all peer countries, 2020 Q1-{GH_LABEL}"

    trend_peer_groups = ["Developers", "Structural", "Regional", "Aspirational"]
    trend_peer_codes = [c for g in trend_peer_groups for c in GH_PEER_GROUPS.get(g, [])]
    trend_codes = [COUNTRY_CODE] + trend_peer_codes
    group_of = {
        c: f"{g} peers" for g in trend_peer_groups for c in GH_PEER_GROUPS.get(g, [])
    }
    group_of[COUNTRY_CODE] = "Algeria"
    label_codes = set(ALL_TREND_LABEL_CODES) | set(GH_PEER_GROUPS.get("Developers", []))

    d = data.loc[data["iso2_code"].isin(trend_codes)].copy()
    d["peer_group"] = d["iso2_code"].map(group_of)
    d.loc[d["iso2_code"] == "KE", "peer_group"] = "Kenya"
    d["country_name"] = (
        d["iso2_code"]
        .map(COUNTRY_NAMES)
        .fillna(d["iso2_code"].map(country_name_from_iso2))
    )
    latest_quarter = d["quarter_start"].max()
    labels = d.loc[
        (d["quarter_start"] == latest_quarter) & d["iso2_code"].isin(label_codes)
    ].sort_values(value_col)
    # Keep end labels at least 15px apart (Kenya, Algeria and Egypt end close
    # together). Algeria's label stays on its line; labels above it move up and
    # labels below it move down. The y-domain is about the data max, rounded up;
    # 1.1 errs toward more space.
    min_gap = 15 * d[value_col].max() * 1.1 / height
    label_y = labels[value_col].tolist()
    dz = labels["iso2_code"].tolist().index(COUNTRY_CODE)
    for i in range(dz + 1, len(label_y)):
        label_y[i] = max(label_y[i], label_y[i - 1] + min_gap)
    for i in range(dz - 1, -1, -1):
        label_y[i] = min(label_y[i], label_y[i + 1] - min_gap)
    labels = labels.assign(label_y=label_y)

    color = alt.Color(
        "peer_group:N",
        title=None,
        scale=alt.Scale(domain=ALL_TREND_GROUP_ORDER, range=ALL_TREND_GROUP_COLORS),
        legend=alt.Legend(orient="bottom", direction="horizontal", columns=3),
    )
    base = alt.Chart(d).encode(
        x=alt.X("quarter_start:T", title=None),
        y=alt.Y(
            f"{value_col}:Q",
            title=y_axis_title,
            axis=alt.Axis(format="~s"),
            scale=alt.Scale(zero=True),
        ),
        color=color,
        detail="country_name:N",
    )
    tooltip = [
        alt.Tooltip("country_name:N", title="Country"),
        alt.Tooltip("peer_group:N", title="Peer group"),
        alt.Tooltip("year:O", title="Year"),
        alt.Tooltip("quarter:O", title="Quarter"),
        alt.Tooltip(f"{value_col}:Q", title=y_axis_title, format=",.0f"),
        alt.Tooltip(
            f"{raw_col}:Q", title=raw_col.replace("_", " ").title(), format=","
        ),
        alt.Tooltip(
            "working_age_population:Q", title="Working-age population", format=",.0f"
        ),
    ]
    peer_lines = (
        base.transform_filter(alt.datum.iso2_code != COUNTRY_CODE)
        .mark_line()
        .encode(tooltip=tooltip)
    )
    algeria_line = (
        base.transform_filter(alt.datum.iso2_code == COUNTRY_CODE)
        .mark_line()
        .encode(tooltip=tooltip)
    )
    latest_labels = (
        alt.Chart(labels)
        .mark_text(align="left", baseline="middle", dx=6, fontSize=FONT_SIZE - 1)
        .encode(
            x="quarter_start:T",
            y=alt.Y("label_y:Q", title=y_axis_title),
            text="country_name:N",
            color=alt.Color(
                "peer_group:N",
                scale=alt.Scale(
                    domain=ALL_TREND_GROUP_ORDER, range=ALL_TREND_GROUP_COLORS
                ),
                legend=None,
            ),
        )
    )
    copilot = pd.DataFrame(
        {
            "quarter_start": [pd.Timestamp("2024-12-15")],
            "label": ["Launch of free access to GitHub Copilot"],
        }
    )
    annotation_rule = (
        alt.Chart(copilot)
        .mark_rule(color="#8A969F", strokeDash=[8, 4], strokeWidth=1.2)
        .encode(x="quarter_start:T")
    )
    annotation_text = (
        alt.Chart(copilot)
        .mark_text(
            align="right",
            baseline="top",
            dx=-6,
            dy=6,
            color="#111111",
            fontSize=FONT_SIZE - 1,
        )
        .encode(x="quarter_start:T", y=alt.value(0), text="label:N")
    )
    return alt.layer(
        peer_lines, algeria_line, annotation_rule, annotation_text, latest_labels
    ).properties(
        width=width, height=height, title=alt.Title(text=title, subtitle=subtitle)
    )


all_developers_trend = make_all_peer_trend(
    developers, raw_col="developers", metric_label="Developers"
)
attaviz.add_caption(
    all_developers_trend,
    "Source: GitHub Innovation Graph data and World Bank SP.POP.1564.TO",
)

alt.VConcatChart(...)

In [15]:
# LinkedIn relative penetration of Disruptive Tech Skills, Algeria vs comparators.
# Morocco is excluded from the comparator set (project decision).
LINKEDIN_FILE = DATA_DIR / "LinkedIn" / "Skill Genome and Skills Pen 2025.xlsx"
LINKEDIN_COMPARATORS = [
    "Algeria",
    "Chile",
    "Poland",
    "Malaysia",
    "Turkiye",
    "Peru",
    "Nigeria",
    "Egypt",
    "Colombia",
    "Tunisia",
    "Romania",
    "Ghana",
    "Iraq",
    "Ecuador",
    "Vietnam",
    "Jordan",
    "Kenya",
]

_skill_pen = (
    pd.read_excel(LINKEDIN_FILE, sheet_name="3A - SPP Ctry", header=5)
    .dropna(subset=["Country", "Skill", "Average", "Global", "Relative"])
    .assign(Country=lambda d: d["Country"].str.strip())
)
tech_skills = (
    _skill_pen.loc[
        lambda df: (
            (df["Skill"] == "Disruptive Tech Skills")
            & df["Country"].isin(LINKEDIN_COMPARATORS)
        )
    ][["Country", "Relative"]]
    .sort_values("Relative", ascending=False)
    .reset_index(drop=True)
)

linkedin_skills_chart = make_bar(
    tech_skills,
    label_col="Country",
    value_col="Relative",
    title="Relative Penetration of Disruptive Tech Skills",
    subtitle="Algeria and comparator countries",
    drop_not_classified=False,
    x_title="Relative penetration",
    fmt=lambda v: f"{v:.2f}",
    highlight=COUNTRY_NAME,
)
attaviz.add_caption(
    linkedin_skills_chart, "Source: LinkedIn Skill Genome and Skills Penetration 2025"
)

alt.VConcatChart(...)

The mix of tech skills in Algeria's Technology, Information and Media industry has also changed ({numref}`fig-box-linkedin-ict-skills`). Telecommunications skills held 5 of the 10 places in 2017 and none after 2020. Software and web skills held 8 places in 2022 and 2023. From 2017 to 2019, telecommunications skills such as GSM and 3G were among the most characteristic skills that LinkedIn members in the industry added. Between 2019 and 2023, web development skills such as JavaScript, CSS, PHP and React.js took their place. In 2024, Generative AI became the second most characteristic skill, and Artificial Intelligence the fifth. These are the first AI skills in the industry's top 10 since 2017.

In [16]:
# LinkedIn Skills Genome (flow): the 10 most characteristic skills added each year
# in Algeria's ICT industry ("Technology, Information and Media").
LINKEDIN_FILE = DATA_DIR / "LinkedIn" / "Skill Genome and Skills Pen 2025.xlsx"
ICT_INDUSTRY = "Technology, Information and Media"
ict_skills = (
    pd.read_excel(LINKEDIN_FILE, sheet_name="2B - SGF Ctry Ind Yr", header=5)
    .dropna(subset=["Country", "Skill"])
    .query("Country == @COUNTRY_NAME and Industry == @ICT_INDUSTRY")
    .filter(["Year", "Skill", "Skill Rank"])
    .rename(columns={"Skill Rank": "rank"})
    .astype({"Year": int, "rank": int})
)

# LinkedIn gives no skill-group tag in this sheet, so tag by hand.
# "Search" (search engines vs executive search) and "Computing" are broad, thus
# both go under software and web skills.
SKILL_FAMILY = {
    "Telecommunications": [
        "Telecommunications",
        "Global System for Mobile Communications (GSM)",
        "3G Technology",
        "Wireless Technologies",
        "Internet Protocol (IP)",
    ],
    "Software and web": [
        "React.js",
        "Web Development",
        "Tailwind CSS",
        "Flutter",
        "JavaScript",
        "Cascading Style Sheets (CSS)",
        "Node.js",
        "Python (Programming Language)",
        "MySQL",
        "PHP",
        "HTML5",
        "Java",
        "Computing",
        "Search",
    ],
    "AI": ["Generative AI", "Artificial Intelligence (AI)"],
    "Language, business and soft skills": [
        "Arabic",
        "French",
        "Resiliency",
        "Business Resilience",
        "Commercial Analysis",
        "Team Management",
        "Client-oriented",
        "Internet Leads",
        "Perseverant",
    ],
}
FAMILY_COLORS = {
    "Telecommunications": "#8A969F",
    "Software and web": ALGERIA_COLOR,
    "AI": "#FF9800",
    "Language, business and soft skills": "#D2D7DB",
}
_family_of = {skill: f for f, skills in SKILL_FAMILY.items() for skill in skills}
ict_skills["family"] = ict_skills["Skill"].map(_family_of)

family_counts = pd.crosstab(ict_skills["Year"], ict_skills["family"]).reindex(
    columns=list(FAMILY_COLORS), fill_value=0
)

# Rank table: years across, ranks down, each cell a skill coloured by family.
# Cells fit about 11 characters, so long names wrap and bracketed names use
# their short form.
SHORT_NAME = {
    "Global System for Mobile Communications (GSM)": "GSM",
    "Cascading Style Sheets (CSS)": "CSS",
    "Artificial Intelligence (AI)": "AI",
    "Internet Protocol (IP)": "IP",
    "Python (Programming Language)": "Python",
    "Telecommunications": "Telecom-\nmunications",
}
ict_table = ict_skills.assign(
    family=lambda d: pd.Categorical(d["family"], categories=list(FAMILY_COLORS)),
    cell=lambda d: d["Skill"].map(
        lambda v: SHORT_NAME.get(
            v, "\n".join(textwrap.wrap(v, 12, break_long_words=False))
        )
    ),
)

_x = alt.X(
    "Year:O",
    title=None,
    scale=alt.Scale(paddingInner=0.06),
    axis=alt.Axis(
        orient="top", labelAngle=0, labelFontWeight="bold", labelColor="#222"
    ),
)
_y = alt.Y(
    "rank:O",
    title=None,
    scale=alt.Scale(paddingInner=0.1),
    axis=alt.Axis(grid=False, labelColor="#222"),
)
_cells = (
    alt.Chart(ict_table)
    .mark_rect(stroke=None)
    .encode(
        x=_x,
        y=_y,
        color=alt.Color(
            "family:N",
            title=None,
            scale=alt.Scale(
                domain=list(FAMILY_COLORS), range=list(FAMILY_COLORS.values())
            ),
            legend=alt.Legend(orient="bottom", columns=2, labelLimit=0),
        ),
        tooltip=[
            alt.Tooltip("Year:O"),
            alt.Tooltip("rank:O", title="Rank"),
            alt.Tooltip("Skill:N"),
            alt.Tooltip("family:N", title="Skill family"),
        ],
    )
)

_names = [
    alt.Chart(ict_table)
    .transform_filter(("" if two else "!") + "test(/\\n/, datum.cell)")
    .mark_text(
        fontSize=FONT_SIZE - 1, lineBreak="\n", lineHeight=13, dy=-6.5 if two else 0
    )
    .encode(
        x=_x,
        y=_y,
        text="cell:N",
        color=alt.condition(
            alt.FieldOneOfPredicate(
                field="family", oneOf=["Language, business and soft skills"]
            ),
            alt.value("#222"),
            alt.value("white"),
        ),
    )
    for two in (False, True)
]
ict_skills_chart = alt.layer(_cells, *_names).properties(
    width=8 * 84,
    height=10 * 40,
    title=alt.Title(
        text=f"Top 10 LinkedIn Skills in {COUNTRY_NAME}'s ICT Industry, 2017-2024",
        subtitle=[
            f"Most characteristic skills added each year in {ICT_INDUSTRY} industry"
        ],
    ),
)
attaviz.add_caption(
    ict_skills_chart,
    "Source: LinkedIn Skill Genome and Skills Penetration 2025\n"
    "Note: The skill families are classified manually based on the nature of each skill.\n"
    "Short names: GSM (Global System for Mobile Communications), CSS (Cascading Style Sheets),\nAI (Artificial Intelligence), "
    "IP (Internet Protocol), Python (Programming Language).",
)

alt.VConcatChart(...)

Egypt, Morocco, Nigeria, Ghana, Kenya are all projected to add millions of developers in the coming years {cite:p}`github2025octoverse`. Developers in Algeria use similar software languages compared to what is used in these countries, with web markup and styling languages being the most commonly prevalent ({numref}`fig-box-languages`). This skillset allows these countries to become an outsourcing destination for software.

GitHub developers in Algeria have long been collaborating with other counterparts in other countries, dominated by those in the European Union and United States ({numref}`fig-box-flows`). Such cross-border software and technical collaboration are considered a form of digital trade {cite:p}`juhasz2026software`. The AI revolution presents Algeria with a strategic opportunity to expand this digital trade, adding a new and high-value sectoral dimension to its economy. Turning Algeria's AI-literate learners into active software builders would let the country take this opportunity.

In [17]:
language_country_codes = [COUNTRY_CODE, "EG", "NG"]
language_country_order = [COUNTRY_NAMES[c] for c in language_country_codes]
language_country_colors = [ALGERIA_COLOR, "#FF9800", "#4EC2C0"]

languages = (
    pd.read_csv(GITHUB_DIR / "languages.csv")
    .loc[lambda d: d["iso2_code"].isin(language_country_codes)]
    .assign(
        quarter_start=lambda d: pd.PeriodIndex(
            d["year"].astype(str) + "Q" + d["quarter"].astype(str), freq="Q"
        ).to_timestamp(),
        country_name=lambda d: d["iso2_code"].map(COUNTRY_NAMES),
    )
)
languages["cluster"] = languages["language"].map(
    clusters.set_index("Language")["Cluster Name"]
)
languages = (
    languages.dropna(subset=["cluster"])
    .groupby(
        ["iso2_code", "country_name", "cluster", "quarter_start", "year", "quarter"],
        as_index=False,
    )["num_pushers"]
    .sum()
)
languages = attach_latest_population(languages, working_age_population).assign(
    num_pushers_per_100k=lambda d: (
        d["num_pushers"] / d["working_age_population"] * 100_000
    )
)
latest_language_quarter = languages["quarter_start"].max()
latest_language_rows = languages.loc[
    languages["quarter_start"] == latest_language_quarter
].copy()
latest_language_label = (
    f"{latest_language_rows['year'].iloc[0]} Q{latest_language_rows['quarter'].iloc[0]}"
)

top_cluster_names = (
    latest_language_rows.sort_values(
        ["country_name", "num_pushers_per_100k"], ascending=[True, False]
    )
    .groupby("iso2_code", as_index=False, group_keys=False)
    .head(10)["cluster"]
    .drop_duplicates()
    .tolist()
)
language_comparison = latest_language_rows.loc[
    lambda d: d["cluster"].isin(top_cluster_names)
].copy()
cluster_sort = (
    language_comparison.groupby("cluster", as_index=False)["num_pushers_per_100k"]
    .sum()
    .sort_values("num_pushers_per_100k", ascending=False)["cluster"]
    .tolist()
)

languages_chart = make_bar(
    language_comparison,
    label_col="cluster",
    value_col="num_pushers_per_100k",
    title="Top GitHub Language Clusters in Algeria, Egypt and Nigeria",
    subtitle=[
        "Each country's top 10 clusters by pushers per 100k working-age population",
        f"Based on {latest_language_label} data",
    ],
    drop_not_classified=False,
    x_title="Pushers per 100k working-age population",
    fmt=lambda v: f"{v:,.1f}",
    group_col="country_name",
    colors=dict(zip(language_country_order, language_country_colors)),
    sort=cluster_sort,
)
attaviz.add_caption(languages_chart, "Source: GitHub Innovation Graph languages data")

alt.VConcatChart(...)

In [18]:
flows_collaborators = (
    pd.read_csv(GITHUB_DIR / "economy_collaborators.csv")
    .loc[lambda d: (d["source"] == COUNTRY_CODE) | (d["destination"] == COUNTRY_CODE)]
    .assign(
        quarter_start=lambda d: pd.PeriodIndex(
            d["year"].astype(str) + "Q" + d["quarter"].astype(str), freq="Q"
        ).to_timestamp()
    )
)

EUROPE_ISO2 = {
    "AL",
    "AD",
    "AT",
    "BY",
    "BE",
    "BA",
    "BG",
    "HR",
    "CY",
    "CZ",
    "DK",
    "EE",
    "FI",
    "FR",
    "DE",
    "GR",
    "HU",
    "IS",
    "IE",
    "IT",
    "XK",
    "LV",
    "LI",
    "LT",
    "LU",
    "MT",
    "MD",
    "MC",
    "ME",
    "NL",
    "MK",
    "NO",
    "PL",
    "PT",
    "RO",
    "RU",
    "SM",
    "RS",
    "SK",
    "SI",
    "ES",
    "SE",
    "CH",
    "UA",
    "GB",
    "VA",
}
_excluded_partners = EUROPE_ISO2 - {"EU"}
_required_partners = ["US", "EU"]


def prepare_algeria_collaborator_flow(flow):
    if flow == "Outbound":
        base = flows_collaborators.loc[lambda d: d["source"] == COUNTRY_CODE].copy()
        partner_col = "destination"
    else:
        base = flows_collaborators.loc[
            lambda d: d["destination"] == COUNTRY_CODE
        ].copy()
        partner_col = "source"
    base = (
        base.loc[lambda d: ~d[partner_col].isin(_excluded_partners)]
        .rename(columns={partner_col: "partner"})
        .groupby(["quarter_start", "year", "quarter", "partner"], as_index=False)[
            "weight"
        ]
        .sum()
    )
    latest_q = base.loc[base["year"] == 2025, "quarter_start"].max()
    latest_label = (
        base.loc[base["quarter_start"] == latest_q, ["year", "quarter"]]
        .drop_duplicates()
        .assign(
            label=lambda d: d["year"].astype(str) + " Q" + d["quarter"].astype(str)
        )["label"]
        .iloc[0]
    )
    top_other = (
        base.loc[lambda d: d["quarter_start"] == latest_q]
        .groupby("partner", as_index=False)["weight"]
        .sum()
        .loc[lambda d: ~d["partner"].isin(_required_partners)]
        .sort_values("weight", ascending=False)
        .head(3)["partner"]
        .tolist()
    )
    selected = _required_partners + top_other
    flow_data = base.loc[lambda d: d["partner"].isin(selected)].assign(
        partner_name=lambda d: d["partner"].map(country_name_from_iso2),
        partner_order=lambda d: d["partner"].map(
            {c: i for i, c in enumerate(selected)}
        ),
        flow=flow,
    )
    order = (
        flow_data.drop_duplicates("partner")
        .sort_values("partner_order")["partner_name"]
        .tolist()
    )
    return flow_data, order, latest_label


def make_collaborator_area_panel(flow, title, width=540, height=300):
    flow_data, order, latest_label = prepare_algeria_collaborator_flow(flow)
    return (
        alt.Chart(flow_data)
        .mark_area(opacity=0.82)
        .encode(
            x=alt.X("quarter_start:T", title=None),
            y=alt.Y(
                "weight:Q",
                title="Collaboration weight",
                stack="zero",
                axis=alt.Axis(format="~s"),
            ),
            color=alt.Color(
                "partner_name:N",
                title="Partner",
                scale=alt.Scale(domain=order),
                legend=alt.Legend(orient="bottom", columns=3),
            ),
            order=alt.Order("partner_order:Q", sort="ascending"),
            tooltip=[
                alt.Tooltip("partner_name:N", title="Partner"),
                alt.Tooltip("year:O", title="Year"),
                alt.Tooltip("quarter:O", title="Quarter"),
                alt.Tooltip("weight:Q", title="Collaboration weight", format=","),
            ],
        )
        .properties(
            width=width,
            height=height,
            title=alt.Title(
                text=title,
                subtitle=f"US, EU, and top three non-European partners in {latest_label}",
            ),
        )
    )


outbound_flows_chart = make_collaborator_area_panel(
    "Outbound", "Outbound GitHub collaborator flows from Algeria"
)
attaviz.add_caption(
    outbound_flows_chart, "Source: GitHub Innovation Graph economy collaborators data"
)

alt.VConcatChart(...)

## References

```{bibliography}
:filter: docname in docnames or key == "chatterji2025chatgpt"
```

[^1]: World Bank analysis using Ookla Speedtest data from Ookla for Good https://datapartnership.org/algeria-economic-monitoring/notebooks/ookla-analysis/ookla-visualization-comparison.html

[^2]: OpenAI Signals data, version 3.0, {cite:p}`openaiecon2026signalsdata`. The rank is calculated by OpenAI from the per-capita share of ChatGPT messages in each country, where rank 1 is the highest. It covers April to June 2026 and is based on a sample of 1,000,000 consumer messages per month.

[^3]: Rank on the Anthropic AI Usage Index (`usage_per_capita_index`) for May 2026, as shown on the Anthropic Economic Index website and released in {cite:p}`massenkoff2026cadences`. The index is a country's share of Claude.ai usage divided by its share of the world's working-age (15 to 64) population. A value of 1.0 means usage is proportional to population. The rank is out of 121 countries. Algeria's index is 0.56, and two other countries have the same value, so the data alone places Algeria between 81st and 83rd. The website shows 82nd.

[^4]: The expected level comes from a regression across 119 countries of log Claude AI Usage Index on log GitHub developers per 100,000 working-age people (second quarter of 2026) and log GDP per capita (PPP, constant 2021 international dollars, World Bank). Algeria's Claude usage is 1.07 times the expected level in May 2026 and 0.88 times in April 2026. On the same basis, with GDP per capita alone, Algeria's developer density is 0.82 times the level its income predicts. Pushes per developer are git pushes divided by developers in the second quarter of 2026 (GitHub Innovation Graph).